# M3 接力续训（v17）——从 HF 最新 backup 续跑到累计 16000

In [ ]:
# ===== v17 接力续训：修复 backup 扫描路径深度过滤 bug（v16 误判无备份） =====
# 设计原则：训练是前台进程，任何崩溃的 traceback 直接出现在批处理日志里，无静默失败。
# 无隧道/sshd/watchdog——监控走 ntfy 信标（topic smolvla-ep-b7Vq2mN4tX）。
import os, re, glob, shutil, time, subprocess, urllib.request

W = "/kaggle/working"
TOPIC = "smolvla2-ep-Kp9wQ4tRz7"
CODE_SHA = "146d8474932c8b566717eb2738ba459337ab71c4"
REV = "a1aaacb7f6cd6ee5fb43120f673cebb0cfea7dd4"
HUB_SHA = "5e8d12a6e2975b0e5e5fce7c8caf47c371d257b6"
os.environ["HF_HOME"] = W + "/hf_cache"
os.makedirs(W + "/logs", exist_ok=True)

def run(cmd, timeout=1800):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, timeout=timeout)
    return (r.stdout + r.stderr).strip()

def beacon(msg):
    try:
        urllib.request.urlopen(urllib.request.Request("https://ntfy.sh/" + TOPIC, data=str(msg).encode()), timeout=8)
    except Exception as e:
        print("[beacon fail]", repr(e), flush=True)

# ---- 1) lerobot 安装（固定 commit）+ dtype 补丁 ----
def have_lerobot():
    return "0.6.2" in run("python -c 'import lerobot; print(lerobot.__version__)'")
if not have_lerobot():
    for cmd in ('uv pip install --system "lerobot[smolvla,libero] @ git+https://github.com/huggingface/lerobot.git@' + CODE_SHA + '"',
                'pip install --no-input "lerobot[smolvla,libero] @ git+https://github.com/huggingface/lerobot.git@' + CODE_SHA + '"'):
        print("[1] install ...", flush=True)
        print(run(cmd, timeout=1200)[-600:], flush=True)
        if have_lerobot():
            break
    if not have_lerobot():
        beacon("FAIL-install")
        raise RuntimeError("lerobot 安装失败（两种方式均失败，输出见上）")
run('sed -i \'s/torch_dtype="bfloat16"/torch_dtype="float32"/\' /usr/local/lib/python3.13/dist-packages/lerobot/policies/smolvla/smolvlm_with_expert.py')
ok = "float32" in run("grep torch_dtype /usr/local/lib/python3.13/dist-packages/lerobot/policies/smolvla/smolvlm_with_expert.py")
print("[1] lerobot 0.6.2 + dtype patch:", ok, flush=True)
if not ok:
    beacon("FAIL-dtype")
    raise RuntimeError("dtype 补丁失败")
beacon("OK-install")

# ---- 2) 从 HF 拉取 8000 步热启动权重（Kaggle Secrets 提供 token）----
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import snapshot_download, list_repo_files
_files = [f for f in list_repo_files("A2bao/smolvla-libero-spatial", repo_type="model", token=os.environ.get("HF_TOKEN", ""))
          if f.startswith("checkpoints-backup/") and len(f.split("/")) >= 3]
_bks = sorted({int(f.split("/")[1]) for f in _files})
if _bks:
    STEP0 = _bks[-1]
    CKPT = snapshot_download("A2bao/smolvla-libero-spatial",
                             allow_patterns=["checkpoints-backup/%07d/*" % STEP0],
                             local_dir=W + "/ckpt_dl") + "/checkpoints-backup/%07d" % STEP0
else:
    STEP0 = 0
    CKPT = snapshot_download("A2bao/smolvla-libero-spatial",
                             allow_patterns=["checkpoint-008000/*"],
                             local_dir=W + "/ckpt_dl") + "/checkpoint-008000"
REMAIN = 8000 - STEP0
print("[2] 接力起点: 累计 %d 步, 本次再训 %d 步" % (STEP0, REMAIN), flush=True)
# checkpoint 里 trainer 写的 processor json 引用本地 tokenizer 子目录（抢救时丢失）→
# 用 Hub base 的 4 个 processor 文件覆盖（tokenizer 步骤改从 SmolVLM2 hub 在线加载，归一化统计量一致，
# 与 v11 成功训练时的加载状态完全等价）
from huggingface_hub import hf_hub_download
HUB_SHA = "5e8d12a6e2975b0e5e5fce7c8caf47c371d257b6"
for fn in ("policy_preprocessor.json",
           "policy_preprocessor_step_5_normalizer_processor.safetensors",
           "policy_postprocessor.json",
           "policy_postprocessor_step_0_unnormalizer_processor.safetensors"):
    shutil.copy(hf_hub_download("lerobot/smolvla_base", fn, revision=HUB_SHA), CKPT + "/")
for f in ("model.safetensors", "train_config.json"):
    if not os.path.exists(CKPT + "/" + f):
        beacon("FAIL-ckpt-incomplete")
        raise RuntimeError("checkpoint 缺文件: " + f)
print("[2] 热启动权重就绪:", CKPT, flush=True)
beacon("OK-ckpt")
# ---- 5) 数据集缓存修复（pinned revision，3 次重试）----
open("/tmp/dsfix.py", "w").write(
    "from huggingface_hub import snapshot_download\n"
    "snapshot_download('lerobot/libero', repo_type='dataset', revision='" + REV + "', max_workers=4)\n"
    "print('DS_OK')\n")
for attempt in range(3):
    out = run("python /tmp/dsfix.py 2>&1 | tail -2", timeout=2400)
    if "DS_OK" in out:
        break
    print("[5] 数据集修复重试", attempt + 1, ":", out[-200:], flush=True)
    time.sleep(20)
else:
    beacon("FAIL-dataset")
    raise RuntimeError("数据集缓存修复失败: " + out[-300:])
print("[5] 数据集缓存完整（revision " + REV[:8] + "）", flush=True)
beacon("OK-setup")

# ---- 6) 磁盘保护：只保留最新 2 个 checkpoint ----
open(W + "/prune.sh", "w").write(
    '#!/bin/bash\nD=' + W + '/outputs/m3_cont16k/checkpoints\n'
    'while true; do sleep 300; [ -d "$D" ] || continue; '
    'ls "$D" | grep -E "^[0-9]+$" | sort -n | head -n -2 | while read c; do rm -rf "$D/$c"; done; done\n')
subprocess.Popen("bash " + W + "/prune.sh", shell=True, start_new_session=True)

# ---- 7) 心跳：每 10 分钟推送进度/GPU/磁盘 ----
HB = ('#!/bin/bash\nexport PATH=/opt/bin:$PATH\nexport LD_LIBRARY_PATH=/usr/local/nvidia/lib64:$LD_LIBRARY_PATH\n'
      'while true; do sleep 600\n'
      'S=$(tail -c 400 ' + W + '/logs/train.log | tr "\\r" "\\n" | grep "Training:" | tail -1)\n'
      'G=$(nvidia-smi --query-gpu=utilization.gpu --format=csv,noheader 2>/dev/null | head -1)\n'
      'D=$(df -BG /kaggle/working | tail -1 | awk "{print \\$4}")\n'
      'curl -s -m 8 -d "HB $S | gpu=$G | disk_free=$D" https://ntfy.sh/' + TOPIC + ' >/dev/null\ndone\n')
open(W + "/hb.sh", "w").write(HB)
subprocess.Popen("bash " + W + "/hb.sh", shell=True, start_new_session=True)

# ---- 8) 前台训练：崩溃自动 resume 重试，traceback 直接进批处理日志 ----
ENV = ("export PATH=/opt/bin:$PATH; export LD_LIBRARY_PATH=/usr/local/nvidia/lib64:$LD_LIBRARY_PATH; "
       "export MUJOCO_GL=egl; export HF_HOME=" + W + "/hf_cache;")
CMD = ("--policy.path=" + CKPT + " --policy.push_to_hub=false "
       "--policy.input_features=null --policy.output_features=null "
       "--dataset.repo_id=lerobot/libero --dataset.revision=" + REV + " "
       "--batch_size=32 --num_workers=2 --accelerator.mixed_precision=fp16 "
       "--accelerator.gradient_accumulation.steps=1 --steps=" + str(REMAIN) + " "
       "--policy.scheduler_warmup_steps=2000 --policy.scheduler_decay_steps=32000 "
       "--save_freq=1000 --seed=1001 "
       "--output_dir=" + W + "/outputs/m3_cont16k --job_name=m3_cont16k")
CKPTD = W + "/outputs/m3_cont16k/checkpoints"
# ---- 7.5) checkpoint 后台保险：落盘即全量上传 HF（含 training_state），配额死亡也保住进度 ----
import threading
from huggingface_hub import HfApi
_HF = HfApi()
_upped = set()
def _ck_uploader():
    while True:
        try:
            for d in sorted(glob.glob(CKPTD + "/0*")):
                name = os.path.basename(d)
                if name in _upped: continue
                if not (os.path.isfile(d + "/pretrained_model/model.safetensors")
                        and os.path.isdir(d + "/training_state")
                        and glob.glob(d + "/training_state/*")): continue
                newest = max((os.path.getmtime(os.path.join(r, f)) for r, _, fs in os.walk(d) for f in fs), default=0)
                if time.time() - newest < 90: continue
                _HF.upload_folder(folder_path=d, path_in_repo="checkpoints-backup/%07d" % (STEP0 + int(name)),
                                  repo_id="A2bao/smolvla-libero-spatial", repo_type="model",
                                  token=os.environ.get("HF_TOKEN", ""))
                _upped.add(name)
                print("[7.5] 已备份 " + name + " -> HF/checkpoints-backup/", flush=True)
        except Exception as e:
            print("[7.5] 备份异常:", repr(e)[:120], flush=True)
        time.sleep(60)
threading.Thread(target=_ck_uploader, daemon=True).start()
# ---- 7.5) [v14 移除后台备份线程；结束时单次上传] ----
MAX_TRY = 2
for attempt in range(1, MAX_TRY + 1):
    cks = sorted(int(d) for d in os.listdir(CKPTD) if d.isdigit()) if os.path.isdir(CKPTD) else []
    if not cks:
        shutil.rmtree(W + "/outputs/m3_cont16k", ignore_errors=True)  # 无 checkpoint 的遗留目录会触发 FileExistsError
    cmd = ENV + " torchrun --standalone --nproc-per-node=2 $(which lerobot-train) " + CMD + ((" --resume=true --config_path=" + CKPTD + "/%08d/pretrained_model" % cks[-1]) if cks else "")
    print("[8] 训练尝试 #%d (resume_ckpt=%s)" % (attempt, cks[-1] if cks else None), flush=True)
    ret = subprocess.call("bash -c 'set -o pipefail; " + cmd + " 2>&1 | tee -a " + W + "/logs/train.log'", shell=True)
    if ret == 0:
        beacon("DONE 16k热启动完成 ret=0")
        print("[8] 训练完成（8000 updates @ eff-bs64, DDP×2）", flush=True)
        break
    beacon("CRASH try%d ret=%d" % (attempt, ret))
    print("[8] 训练异常退出 ret=%d，30s 后 resume 重启" % ret, flush=True)
    time.sleep(30)
else:
    beacon("GIVEUP after 30 tries")
    raise RuntimeError("训练连续 %d 次失败，放弃（详见上方 train.log 输出）" % MAX_TRY)
# ---- 9) 上传 16000 checkpoint 到 HF（这次连 training_state 一起，不再丢断点）----
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import upload_folder
final_ckpt = W + "/outputs/m3_cont16k/checkpoints/016000"
if os.path.isdir(final_ckpt):
    upload_folder(folder_path=final_ckpt, path_in_repo="checkpoint-%07d" % (STEP0 + 8000),
                  repo_id="A2bao/smolvla-libero-spatial", repo_type="model",
                  token=os.environ["HF_TOKEN"])
    print("[9] 累计 %d 步 checkpoint 已上传 HF（含 training_state）" % (STEP0 + 8000), flush=True)
    beacon("HF-UPLOAD-DONE")
print("ALL DONE", flush=True)
